In [1]:
import os
import numpy as np
import pandas as pd
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
files = ['two_class_raw_2s_yo_0.5.csv', 'two_class_raw_3s_no.csv', 'two_class_raw_4s_no.csv','two_class_raw_5s_no.csv', 'two_class_raw_5s_yo_0.5.csv']

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/improved_features_data_main/'

In [ ]:
# file_results = {}
# for file in tqdm(files, desc="Tuning hyperparameters"):
#     data_path = os.path.join(base_path, file)
#     features = pd.read_csv(data_path)
#     features.drop(['center_time', 'start_time', 'end_time'], axis=1, inplace=True)
#     details = file.split('_')
#     exp_name = f"{details[3]}_{details[-1].replace('.csv', '')}"
#     print(f"Analyzing {exp_name}")
    
#     # split data
#     X = features.drop(columns=['label', 'experiment_id'])
#     y = features['label']
#     groups = features['experiment_id']

#     splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
#     train_idx, test_idx = next(splitter.split(X, y, groups))

#     X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
#     y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
#     groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]
    
#     X_train_resampled, y_train_resampled = sampler.fit_resample(X_train, y_train)
#     X_train_resampled = pd.DataFrame(X_train_resampled, columns=X_train_fold.columns)
#     y_train_resampled = pd.Series(y_train_resampled)
    
    
#     model = DecisionTreeClassifier(
#         random_state=42,
#     )
    
#     # Encode labels
#     label_encoder = LabelEncoder()
#     y_train_encoded = label_encoder.fit_transform(y_train_resampled)

In [3]:
def get_search_spaces():
    """Define hyperparameter search spaces for each model"""
    
    # Decision Tree search space
    dt_space = {
        'max_depth': hp.choice('dt_max_depth', [None, 3, 5, 7, 10, 15, 20]),
        'min_samples_split': hp.choice('dt_min_samples_split', [2, 5, 10, 20]),
        'min_samples_leaf': hp.choice('dt_min_samples_leaf', [1, 2, 4, 8]),
        'criterion': hp.choice('dt_criterion', ['gini', 'entropy']),
        'max_features': hp.choice('dt_max_features', ['sqrt', 'log2', None]),
        'class_weight': hp.choice('dt_class_weight', ['balanced', None])
    }
    
    # Random Forest search space
    rf_space = {
        'n_estimators': hp.choice('rf_n_estimators', [50, 100, 200, 300, 500]),
        'max_depth': hp.choice('rf_max_depth', [None, 3, 5, 7, 10, 15, 20]),
        'min_samples_split': hp.choice('rf_min_samples_split', [2, 5, 10, 20]),
        'min_samples_leaf': hp.choice('rf_min_samples_leaf', [1, 2, 4, 8]),
        'max_features': hp.choice('rf_max_features', ['sqrt', 'log2', None]),
        'bootstrap': hp.choice('rf_bootstrap', [True, False]),
        'class_weight': hp.choice('rf_class_weight', ['balanced', 'balanced_subsample', None])
    }
    
    # XGBoost search space
    xgb_space = {
        'n_estimators': hp.choice('xgb_n_estimators', [100, 200, 300, 500, 800]),
        'max_depth': hp.choice('xgb_max_depth', [3, 4, 5, 6, 7, 8]),
        'learning_rate': hp.uniform('xgb_learning_rate', 0.01, 0.3),
        'subsample': hp.uniform('xgb_subsample', 0.6, 1.0),
        'colsample_bytree': hp.uniform('xgb_colsample_bytree', 0.6, 1.0),
        'reg_alpha': hp.uniform('xgb_reg_alpha', 0, 1),
        'reg_lambda': hp.uniform('xgb_reg_lambda', 0, 1),
        'min_child_weight': hp.choice('xgb_min_child_weight', [1, 3, 5, 7]),
        'gamma': hp.uniform('xgb_gamma', 0, 0.5),
        'scale_pos_weight': hp.uniform('xgb_scale_pos_weight', 1, 10)
    }
    
    return dt_space, rf_space, xgb_space

In [4]:
def create_objective_function_with_cv_smote(X_train, y_train, groups_train, model_type, cv_folds=5):
    """
    Create objective function that applies SMOTE-Tomek within each CV fold
    This prevents data leakage and gives more robust hyperparameter selection
    """
    
    def objective(params):
        try:
            # Initialize GroupKFold
            gkf = GroupKFold(n_splits=cv_folds)
            fold_scores = []
            
            # Perform cross-validation
            for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
                
                # Split data for this fold
                X_fold_train = X_train.iloc[train_idx]
                X_fold_val = X_train.iloc[val_idx] 
                y_fold_train = y_train.iloc[train_idx]
                y_fold_val = y_train.iloc[val_idx]
                
                # Apply SMOTE-Tomek to training fold only
                smote_tomek = SMOTETomek(random_state=42)
                X_fold_resampled, y_fold_resampled = smote_tomek.fit_resample(
                    X_fold_train, y_fold_train
                )
                
                # Initialize model with current hyperparameters
                if model_type == 'dt':
                    model = DecisionTreeClassifier(random_state=42, **params)
                elif model_type == 'rf':
                    model = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
                elif model_type == 'xgb':
                    model = XGBClassifier(
                        random_state=42, 
                        eval_metric='logloss', 
                        verbosity=0, 
                        **params
                    )
                
                # Train on resampled fold training data
                model.fit(X_fold_resampled, y_fold_resampled)
                
                # Predict on fold validation data (original, unbalanced)
                y_pred = model.predict(X_fold_val)
                
                # Calculate F1 score for this fold
                fold_score = f1_score(y_fold_val, y_pred, average='weighted')
                fold_scores.append(fold_score)
            
            # Return negative mean CV score (hyperopt minimizes)
            mean_cv_score = np.mean(fold_scores)
            return {'loss': -mean_cv_score, 'status': STATUS_OK}
            
        except Exception as e:
            # Return high loss for invalid parameter combinations
            return {'loss': 1, 'status': STATUS_OK}
    
    return objective

In [11]:
# Main tuning function
def tune_hyperparameters(X_train, y_train, groups_train, model_type, max_evals=100, cv_folds=5):
    """
    Tune hyperparameters using GroupKFold CV with SMOTE-Tomek applied within each fold
    
    Parameters:
    - X_train, y_train, groups_train: Training data
    - model_type: 'dt', 'rf', or 'xgb'
    - max_evals: Maximum number of hyperopt evaluations
    - cv_folds: Number of CV folds
    
    Returns:
    - best_params: Best hyperparameters found
    - best_cv_score: Best cross-validation score
    - trials: Hyperopt trials object
    """
    
    print(f"    Training set: {len(X_train)} samples, {len(groups_train.unique())} groups")
    print(f"    Class distribution: {dict(y_train.value_counts())}")
    print(f"    Using {cv_folds}-fold GroupKFold CV with SMOTE-Tomek per fold")
    
    # Get search space
    dt_space, rf_space, xgb_space = get_search_spaces()
    
    if model_type == 'dt':
        search_space = dt_space
    elif model_type == 'rf':  
        search_space = rf_space
    elif model_type == 'xgb':
        search_space = xgb_space
    else:
        raise ValueError("model_type must be 'dt', 'rf', or 'xgb'")
    
    # Create objective function
    objective = create_objective_function_with_cv_smote(
        X_train, y_train, groups_train, model_type, cv_folds
    )
    
    # Run hyperopt optimization
    trials = Trials()
    best = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        # random_state=42,
        verbose=False
    )
    
    # Get best CV score
    best_cv_score = -min([trial['result']['loss'] for trial in trials.trials])
    
    return best, best_cv_score, trials

In [6]:
# Function to train final model and evaluate on test set
def train_final_model(X_train, y_train, X_test, y_test, model_type, best_params):
    """
    Train final model with best parameters on full training set and evaluate on test set
    """
    
    print(f"    Training final {model_type.upper()} model...")
    
    # Apply SMOTE-Tomek to full training set
    smote_tomek = SMOTETomek(random_state=42)
    X_train_resampled, y_train_resampled = smote_tomek.fit_resample(X_train, y_train)
    
    print(f"    After SMOTE-Tomek: {len(X_train_resampled)} samples")
    print(f"    Resampled class distribution: {dict(pd.Series(y_train_resampled).value_counts())}")
    
    # Initialize model with best parameters
    if model_type == 'dt':
        model = DecisionTreeClassifier(random_state=42, **best_params)
    elif model_type == 'rf':
        model = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)  
    elif model_type == 'xgb':
        model = XGBClassifier(
            random_state=42, 
            eval_metric='logloss', 
            verbosity=0, 
            **best_params
        )
    
    # Train on full resampled training data
    model.fit(X_train_resampled, y_train_resampled)
    
    # Predict on test set
    y_pred = model.predict(X_test)
    
    # Calculate test score
    test_score = f1_score(y_test, y_pred, average='weighted')
    
    return model, test_score

In [8]:
# Complete pipeline for single dataset
def tune_single_dataset(X, y, groups, dataset_name, max_evals=100, cv_folds=5, test_size=0.2):
    """
    Complete tuning pipeline for a single dataset using CV approach
    """
    
    print(f"\n=== Processing {dataset_name} ===")
    print(f"Total samples: {len(X)}")
    print(f"Total groups: {len(groups.unique())}")
    print(f"Class distribution: {dict(y.value_counts())}")
    
    # Split into train and test sets using GroupShuffleSplit
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups))
    
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]
    
    print(f"Train: {len(X_train)} samples, {len(groups_train.unique())} groups")
    print(f"Test: {len(X_test)} samples, {len(groups_test.unique())} groups")
    
    models = ['dt', 'rf', 'xgb']
    results = {}
    
    for model_type in models:
        print(f"\n  Tuning {model_type.upper()} with {cv_folds}-fold CV...")
        
        # Tune hyperparameters using CV on training set
        best_params, best_cv_score, trials = tune_hyperparameters(
            X_train, y_train, groups_train, model_type, max_evals, cv_folds
        )
        
        print(f"    Best CV F1: {best_cv_score:.4f}")
        
        # Train final model and evaluate on test set
        final_model, test_score = train_final_model(
            X_train, y_train, X_test, y_test, model_type, best_params
        )
        
        print(f"    Test F1: {test_score:.4f}")
        
        # Store results
        results[model_type] = {
            'best_params': best_params,
            'best_cv_score': best_cv_score,
            'test_score': test_score,
            'trials': trials,
            'final_model': final_model
        }
    
    return results, {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'groups_train': groups_train, 'groups_test': groups_test
    }

In [9]:
# split data
data_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/improved_features_data_main/two_class_raw_3s_no.csv'
features = pd.read_csv(data_path)
features.drop(['center_time', 'start_time', 'end_time'], axis=1, inplace=True)
X = features.drop(columns=['label', 'experiment_id'])
y = features['label']
groups = features['experiment_id']

dataset_name = '3s_no'

In [12]:
tune_single_dataset(X, y, groups, dataset_name, max_evals=100, cv_folds=5, test_size=0.2)


=== Processing 3s_no ===
Total samples: 789
Total groups: 41
Class distribution: {'non-void': np.int64(466), 'void': np.int64(323)}
Train: 578 samples, 32 groups
Test: 211 samples, 9 groups

  Tuning DT with 5-fold CV...
    Training set: 578 samples, 32 groups
    Class distribution: {'non-void': np.int64(336), 'void': np.int64(242)}
    Using 5-fold GroupKFold CV with SMOTE-Tomek per fold
    Best CV F1: 0.7201
    Training final DT model...
    After SMOTE-Tomek: 598 samples
    Resampled class distribution: {'non-void': np.int64(299), 'void': np.int64(299)}


TypeError: DecisionTreeClassifier.__init__() got an unexpected keyword argument 'dt_class_weight'

In [ ]:
import numpy as np
import pandas as pd
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

# Define search spaces for each model
def get_search_spaces():
    """Define hyperparameter search spaces for each model"""
    
    # Decision Tree search space
    dt_space = {
        'max_depth': hp.choice('dt_max_depth', [None, 3, 5, 7, 10, 15, 20]),
        'min_samples_split': hp.choice('dt_min_samples_split', [2, 5, 10, 20]),
        'min_samples_leaf': hp.choice('dt_min_samples_leaf', [1, 2, 4, 8]),
        'criterion': hp.choice('dt_criterion', ['gini', 'entropy']),
        'max_features': hp.choice('dt_max_features', ['sqrt', 'log2', None]),
        'class_weight': hp.choice('dt_class_weight', ['balanced', None])
    }
    
    # Random Forest search space
    rf_space = {
        'n_estimators': hp.choice('rf_n_estimators', [50, 100, 200, 300, 500]),
        'max_depth': hp.choice('rf_max_depth', [None, 3, 5, 7, 10, 15, 20]),
        'min_samples_split': hp.choice('rf_min_samples_split', [2, 5, 10, 20]),
        'min_samples_leaf': hp.choice('rf_min_samples_leaf', [1, 2, 4, 8]),
        'max_features': hp.choice('rf_max_features', ['sqrt', 'log2', None]),
        'bootstrap': hp.choice('rf_bootstrap', [True, False]),
        'class_weight': hp.choice('rf_class_weight', ['balanced', 'balanced_subsample', None])
    }
    
    # XGBoost search space
    xgb_space = {
        'n_estimators': hp.choice('xgb_n_estimators', [100, 200, 300, 500, 800]),
        'max_depth': hp.choice('xgb_max_depth', [3, 4, 5, 6, 7, 8]),
        'learning_rate': hp.uniform('xgb_learning_rate', 0.01, 0.3),
        'subsample': hp.uniform('xgb_subsample', 0.6, 1.0),
        'colsample_bytree': hp.uniform('xgb_colsample_bytree', 0.6, 1.0),
        'reg_alpha': hp.uniform('xgb_reg_alpha', 0, 1),
        'reg_lambda': hp.uniform('xgb_reg_lambda', 0, 1),
        'min_child_weight': hp.choice('xgb_min_child_weight', [1, 3, 5, 7]),
        'gamma': hp.uniform('xgb_gamma', 0, 0.5),
        'scale_pos_weight': hp.uniform('xgb_scale_pos_weight', 1, 10)
    }
    
    return dt_space, rf_space, xgb_space

# Objective function with SMOTE-Tomek applied within each CV fold
def create_objective_function_with_cv_smote(X_train, y_train, groups_train, model_type, cv_folds=5):
    """
    Create objective function that applies SMOTE-Tomek within each CV fold
    This prevents data leakage and gives more robust hyperparameter selection
    """
    
    def objective(params):
        try:
            # Initialize GroupKFold
            gkf = GroupKFold(n_splits=cv_folds)
            fold_scores = []
            
            # Perform cross-validation
            for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
                
                # Split data for this fold
                X_fold_train = X_train.iloc[train_idx]
                X_fold_val = X_train.iloc[val_idx] 
                y_fold_train = y_train.iloc[train_idx]
                y_fold_val = y_train.iloc[val_idx]
                
                # Apply SMOTE-Tomek to training fold only
                smote_tomek = SMOTETomek(random_state=42)
                X_fold_resampled, y_fold_resampled = smote_tomek.fit_resample(
                    X_fold_train, y_fold_train
                )
                
                # Initialize model with current hyperparameters
                if model_type == 'dt':
                    model = DecisionTreeClassifier(random_state=42, **params)
                elif model_type == 'rf':
                    model = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
                elif model_type == 'xgb':
                    model = XGBClassifier(
                        random_state=42, 
                        eval_metric='logloss', 
                        verbosity=0, 
                        **params
                    )
                
                # Train on resampled fold training data
                model.fit(X_fold_resampled, y_fold_resampled)
                
                # Predict on fold validation data (original, unbalanced)
                y_pred = model.predict(X_fold_val)
                
                # Calculate F1 score for this fold
                fold_score = f1_score(y_fold_val, y_pred, average='weighted')
                fold_scores.append(fold_score)
            
            # Return negative mean CV score (hyperopt minimizes)
            mean_cv_score = np.mean(fold_scores)
            return {'loss': -mean_cv_score, 'status': STATUS_OK}
            
        except Exception as e:
            # Return high loss for invalid parameter combinations
            return {'loss': 1, 'status': STATUS_OK}
    
    return objective

# Main tuning function
def tune_hyperparameters(X_train, y_train, groups_train, model_type, max_evals=100, cv_folds=5):
    """
    Tune hyperparameters using GroupKFold CV with SMOTE-Tomek applied within each fold
    
    Parameters:
    - X_train, y_train, groups_train: Training data
    - model_type: 'dt', 'rf', or 'xgb'
    - max_evals: Maximum number of hyperopt evaluations
    - cv_folds: Number of CV folds
    
    Returns:
    - best_params: Best hyperparameters found
    - best_cv_score: Best cross-validation score
    - trials: Hyperopt trials object
    """
    
    print(f"    Training set: {len(X_train)} samples, {len(groups_train.unique())} groups")
    print(f"    Class distribution: {dict(y_train.value_counts())}")
    print(f"    Using {cv_folds}-fold GroupKFold CV with SMOTE-Tomek per fold")
    
    # Get search space
    dt_space, rf_space, xgb_space = get_search_spaces()
    
    if model_type == 'dt':
        search_space = dt_space
    elif model_type == 'rf':  
        search_space = rf_space
    elif model_type == 'xgb':
        search_space = xgb_space
    else:
        raise ValueError("model_type must be 'dt', 'rf', or 'xgb'")
    
    # Create objective function
    objective = create_objective_function_with_cv_smote(
        X_train, y_train, groups_train, model_type, cv_folds
    )
    
    # Run hyperopt optimization
    trials = Trials()
    best = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        random_state=42,
        verbose=False
    )
    
    # Get best CV score
    best_cv_score = -min([trial['result']['loss'] for trial in trials.trials])
    
    return best, best_cv_score, trials

# Function to train final model and evaluate on test set
def train_final_model(X_train, y_train, X_test, y_test, model_type, best_params):
    """
    Train final model with best parameters on full training set and evaluate on test set
    """
    
    print(f"    Training final {model_type.upper()} model...")
    
    # Apply SMOTE-Tomek to full training set
    smote_tomek = SMOTETomek(random_state=42)
    X_train_resampled, y_train_resampled = smote_tomek.fit_resample(X_train, y_train)
    
    print(f"    After SMOTE-Tomek: {len(X_train_resampled)} samples")
    print(f"    Resampled class distribution: {dict(pd.Series(y_train_resampled).value_counts())}")
    
    # Initialize model with best parameters
    if model_type == 'dt':
        model = DecisionTreeClassifier(random_state=42, **best_params)
    elif model_type == 'rf':
        model = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)  
    elif model_type == 'xgb':
        model = XGBClassifier(
            random_state=42, 
            eval_metric='logloss', 
            verbosity=0, 
            **best_params
        )
    
    # Train on full resampled training data
    model.fit(X_train_resampled, y_train_resampled)
    
    # Predict on test set
    y_pred = model.predict(X_test)
    
    # Calculate test score
    test_score = f1_score(y_test, y_pred, average='weighted')
    
    return model, test_score

# Complete pipeline for single dataset
def tune_single_dataset(X, y, groups, dataset_name, max_evals=100, cv_folds=5, test_size=0.2):
    """
    Complete tuning pipeline for a single dataset using CV approach
    """
    
    print(f"\n=== Processing {dataset_name} ===")
    print(f"Total samples: {len(X)}")
    print(f"Total groups: {len(groups.unique())}")
    print(f"Class distribution: {dict(y.value_counts())}")
    
    # Split into train and test sets using GroupShuffleSplit
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups))
    
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]
    
    print(f"Train: {len(X_train)} samples, {len(groups_train.unique())} groups")
    print(f"Test: {len(X_test)} samples, {len(groups_test.unique())} groups")
    
    models = ['dt', 'rf', 'xgb']
    results = {}
    
    for model_type in models:
        print(f"\n  Tuning {model_type.upper()} with {cv_folds}-fold CV...")
        
        # Tune hyperparameters using CV on training set
        best_params, best_cv_score, trials = tune_hyperparameters(
            X_train, y_train, groups_train, model_type, max_evals, cv_folds
        )
        
        print(f"    Best CV F1: {best_cv_score:.4f}")
        
        # Train final model and evaluate on test set
        final_model, test_score = train_final_model(
            X_train, y_train, X_test, y_test, model_type, best_params
        )
        
        print(f"    Test F1: {test_score:.4f}")
        
        # Store results
        results[model_type] = {
            'best_params': best_params,
            'best_cv_score': best_cv_score,
            'test_score': test_score,
            'trials': trials,
            'final_model': final_model
        }
    
    return results, {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'groups_train': groups_train, 'groups_test': groups_test
    }

# Complete pipeline for all datasets
def run_complete_tuning_pipeline(files, max_evals=100, cv_folds=5):
    """
    Run complete hyperparameter tuning pipeline for all datasets and models
    """
    
    all_results = {}
    
    for file in files:
        print(f"\n{'='*70}")
        
        # Load your data here - you'll need to implement this
        # X, y, groups = load_your_data(file)
        
        # Run tuning for this dataset
        # results, data_splits = tune_single_dataset(X, y, groups, file, max_evals, cv_folds)
        # all_results[file] = results
    
    return all_results

# Function to compare results across datasets
def compare_results(all_results):
    """Compare results across all datasets and models"""
    
    comparison_data = []
    
    for dataset, models in all_results.items():
        for model, metrics in models.items():
            comparison_data.append({
                'Dataset': dataset,
                'Model': model.upper(),
                'CV_F1': metrics['best_cv_score'],
                'Test_F1': metrics['test_score'],
                'CV_Test_Gap': metrics['best_cv_score'] - metrics['test_score'],
                'Window_Size': dataset.split('_')[3],
                'Overlap': 'yes' if 'yo' in dataset else 'no'
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Sort by test score
    comparison_df = comparison_df.sort_values('Test_F1', ascending=False)
    
    return comparison_df

# Function to analyze overfitting
def analyze_overfitting(comparison_df):
    """Analyze overfitting by looking at CV vs Test score gaps"""
    
    print("\n" + "="*50)
    print("OVERFITTING ANALYSIS")
    print("="*50)
    print("Large CV-Test gaps indicate overfitting")
    print(f"Mean CV-Test gap: {comparison_df['CV_Test_Gap'].mean():.4f}")
    print(f"Std CV-Test gap: {comparison_df['CV_Test_Gap'].std():.4f}")
    
    print("\nWorst overfitting cases:")
    worst_overfitting = comparison_df.nlargest(5, 'CV_Test_Gap')[
        ['Dataset', 'Model', 'CV_F1', 'Test_F1', 'CV_Test_Gap']
    ]
    print(worst_overfitting)

# Example usage with CV approach
if __name__ == "__main__":
    files = [
        'two_class_raw_2s_yo_0.5.csv', 
        'two_class_raw_3s_no.csv', 
        'two_class_raw_4s_no.csv',
        'two_class_raw_5s_no.csv', 
        'two_class_raw_5s_yo_0.5.csv'
    ]
    
    # Run complete pipeline with CV
    # all_results = run_complete_tuning_pipeline(files, max_evals=100, cv_folds=5)
    
    # Compare results
    # comparison = compare_results(all_results)
    # print("\n" + "="*70)
    # print("FINAL COMPARISON - TOP 10 RESULTS")
    # print("="*70)
    # print(comparison.head(10))
    
    # Analyze overfitting
    # analyze_overfitting(comparison)
    
    # Find best windowing strategy
    # best_overall = comparison.iloc[0]
    # print(f"\nBEST OVERALL COMBINATION:")
    # print(f"Dataset: {best_overall['Dataset']}")
    # print(f"Model: {best_overall['Model']}")
    # print(f"CV F1 Score: {best_overall['CV_F1']:.4f}")
    # print(f"Test F1 Score: {best_overall['Test_F1']:.4f}")
    # print(f"Overfitting Gap: {best_overall['CV_Test_Gap']:.4f}")
